# VJP-delta

`VJPDelta` fits an additive direction at each chosen source layer. It reads a contrast at a later target layer, then uses a vector-Jacobian product (VJP) to map that contrast back to each source layer. The fitted vectors use the usual state-control additive intervention during generation.

The method is motivated by [J-lens](https://github.com/anthropics/jacobian-lens). The VJP-delta method provenance is [wassname/vjp-steering at `cb03382`](https://github.com/wassname/vjp-steering/tree/cb03382ebd0cc9cad615d169f42e68e8ae3e12a7). This is a repo-native implementation from the documented mathematics and Steerability contracts. It does not copy or assert a license for that source repository.

<!-- PI[gpt-5.6-terra]: added this notebook. -->

## Setup

This CPU demonstration uses the repository's small random Llama fixture. The control is used through `SteeringPipeline`, the same public pipeline API used with a Hugging Face checkpoint.

In [1]:
from pathlib import Path
import tempfile

from steerability.algorithms.core.steering_pipeline import SteeringPipeline
from steerability.algorithms.state_control.vjp_delta import VJPDelta
from steerability.spipe import SPipe
from tests.utils.tiny_models import tiny_llama, wordlevel_tokenizer

model = tiny_llama()
tokenizer = wordlevel_tokenizer()
fit_data = {
    "positives": ["the cat sat", "the dog ran"],
    "negatives": ["dog ran fast"],
}
control = VJPDelta(
    data=fit_data,
    target_layer=2,
    source_layer_ids=[0, 1],
    skip_first=0,
    strength=0.5,
)
pipeline = SteeringPipeline(
    model=model,
    tokenizer=tokenizer,
    controls=[control],
    model_name_or_path="tiny-llama-demo",
)

## Extract and steer

The raw prompts have unequal positive and negative pool sizes. `steer()` extracts the vectors, then binds the standard additive intervention. The stored direction rows are unit norm.

In [2]:
pipeline.steer()
vector = control.export_state()["intervention_0/transform"]
assert set(vector.directions) == {0, 1}
assert all(abs(direction.norm().item() - 1.0) < 1e-5 for direction in vector.directions.values())

reply = pipeline.generate(text="the cat", max_new_tokens=3, do_sample=False)
print({"layers": sorted(vector.directions), "reply": repr(reply)})

{'layers': [0, 1], 'reply': "'the'"}


## Freeze and reload

The frozen form contains the fitted vectors as an `ActivationAdapter`. Reloading it resolves the stored additive artifact and does not run another VJP fit.

In [3]:
bundle = Path(tempfile.mkdtemp()) / "vjp_delta_demo"
saved = pipeline.to_spipe().save(bundle)
reloaded = SPipe.load(saved).pipeline()
assert type(reloaded.state_controls[0]).__name__ == "ActivationAdapter"
reloaded.model, reloaded.tokenizer = model, tokenizer
reloaded.steer()
reloaded_reply = reloaded.generate(text="the cat", max_new_tokens=3, do_sample=False)
assert reloaded_reply == reply
print({"bundle": str(saved), "reply_matches": reloaded_reply == reply})

{'bundle': '/tmp/tmp1jqfr_4f/vjp_delta_demo', 'reply_matches': True}
